# 📊 Multiple Linear Regression - Interactive Learning Notebook

## TTK 4260: Multivariate Data Analysis and ML

---

**Author:** Based on lecture materials by Adil Rasheed, NTNU  
**Interactive Notebook Created:** January 2026

---

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** what a coefficient in multiple regression means (partial effect, "holding others fixed")
2. **Identify** when coefficient interpretations are reliable (role of correlation/independence)
3. **Correctly include** categorical predictors (dummy encoding + baseline) and interpret them
4. **Explain** the effect of scaling on coefficients and on regularization
5. **Compare** L0 / L1 / L2 regularization in terms of sparsity, stability, and interpretability

---

## 📑 Table of Contents

1. [Historical Background](#1-historical-background)
2. [Multiple Regression Model](#2-multiple-regression-model)
3. [Interpretability & Correlated Predictors](#3-interpretability--correlated-predictors)
4. [Categorical Variables in MLR](#4-categorical-variables-in-mlr)
5. [Regularization: L2, L1, L0](#5-regularization-l2-l1-l0)
6. [Feature Scaling](#6-feature-scaling)
7. [Summary & Key Takeaways](#7-summary--key-takeaways)

In [1]:
# ============================================
# SETUP: Install and Import Required Libraries
# ============================================

# First, let's install the required packages
import subprocess
import sys

def install_packages():
    packages = ['numpy', 'pandas', 'plotly', 'scikit-learn', 'scipy', 'statsmodels']
    for package in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    print("✅ All packages installed successfully!")

install_packages()

✅ All packages installed successfully!


In [2]:
# ============================================
# Import Libraries
# ============================================

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ All libraries imported successfully!")
print(f"\n📦 NumPy version: {np.__version__}")
print(f"📦 Pandas version: {pd.__version__}")

✅ All libraries imported successfully!

📦 NumPy version: 2.3.5
📦 Pandas version: 2.3.3


---

# 1. Historical Background

<a id='1-historical-background'></a>

## 📜 Where Did Regression Come From?

### Early 1800s: Astronomy Problems

Regression analysis has its roots in **astronomy**. Scientists like Carl Friedrich Gauss and Adrien-Marie Legendre needed to reconcile noisy measurements of celestial bodies.

- **Least squares** was developed to average out measurement errors
- Famous priority dispute: **Gauss vs. Legendre** (both claimed to have invented it first!)

### Late 1800s: The Word "Regression"

**Francis Galton** studied the relationship between parents' heights and their children's heights. He noticed something interesting:

> *"Children of very tall parents tended to be less tall than their parents."*

He called this phenomenon **"regression toward mediocrity"** (now known as "regression to the mean").

### 💡 Fun Fact
*Statistics began as a way to handle disappointment in data!*

In [3]:
# ============================================
# Interactive Demo: Regression to the Mean
# ============================================

# Simulate Galton's parent-child height study
np.random.seed(42)
n_families = 500

# Generate parent heights (in inches)
parent_heights = np.random.normal(68, 3, n_families)

# Children's heights: correlated with parents but with regression to mean
# Y = intercept + slope * X + noise
# With regression to mean, slope < 1
regression_coefficient = 0.6  # This causes regression to the mean
mean_height = 68

child_heights = mean_height + regression_coefficient * (parent_heights - mean_height) + np.random.normal(0, 2, n_families)

# Create DataFrame
galton_df = pd.DataFrame({
    'Parent Height (inches)': parent_heights,
    'Child Height (inches)': child_heights
})

# Fit regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(parent_heights, child_heights)

# Create the plot
fig = go.Figure()

# Scatter plot
fig.add_trace(go.Scatter(
    x=parent_heights,
    y=child_heights,
    mode='markers',
    marker=dict(size=8, color='royalblue', opacity=0.6),
    name='Families',
    hovertemplate='Parent: %{x:.1f} in<br>Child: %{y:.1f} in<extra></extra>'
))

# Regression line
x_line = np.linspace(parent_heights.min(), parent_heights.max(), 100)
y_line = intercept + slope * x_line

fig.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode='lines',
    line=dict(color='red', width=3),
    name=f'Regression Line (slope = {slope:.2f})'
))

# 45-degree line (perfect inheritance)
fig.add_trace(go.Scatter(
    x=x_line,
    y=x_line,
    mode='lines',
    line=dict(color='gray', width=2, dash='dash'),
    name='Perfect Inheritance (slope = 1)'
))

fig.update_layout(
    title=dict(
        text="<b>Galton's Discovery: Regression to the Mean</b><br><sup>Children of tall parents are tall, but not AS tall (slope < 1)</sup>",
        font=dict(size=18)
    ),
    xaxis_title='Parent Height (inches)',
    yaxis_title='Child Height (inches)',
    width=900,
    height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    hovermode='closest'
)

# Add annotation
fig.add_annotation(
    x=75, y=70,
    text=f"Regression slope = {slope:.2f}<br>If slope were 1, children would be<br>exactly as extreme as parents",
    showarrow=True,
    arrowhead=2,
    bgcolor="lightyellow",
    bordercolor="orange",
    borderwidth=2
)

fig.show()

print(f"\n📊 Regression Analysis Results:")
print(f"   Slope: {slope:.3f} (< 1 indicates regression to the mean)")
print(f"   Intercept: {intercept:.3f}")
print(f"   R-squared: {r_value**2:.3f}")
print(f"\n💡 Interpretation: For every 1 inch increase in parent height,")
print(f"   children are expected to be only {slope:.2f} inches taller than average.")


📊 Regression Analysis Results:
   Slope: 0.550 (< 1 indicates regression to the mean)
   Intercept: 30.684
   R-squared: 0.408

💡 Interpretation: For every 1 inch increase in parent height,
   children are expected to be only 0.55 inches taller than average.


## 🏠 Why Multiple Linear Regression?

Real-world outcomes **rarely depend on a single factor**. Let's consider house prices as a classic example:

### The Problem with Simple Linear Regression

Using only house size to predict price:
$$y = \theta_0 + \theta_1 \cdot \text{Size} + \varepsilon$$

**Limitations:**
- Ignores other important factors
- Two houses with the same size can have very different prices
- Leads to biased or noisy predictions

### The Solution: Multiple Linear Regression

Include ALL relevant features:
$$y = \theta_0 + \theta_1 \cdot \text{Size} + \theta_2 \cdot \text{Bedrooms} + \theta_3 \cdot \text{Age} + \cdots + \varepsilon$$

**House price features might include:**
- Size (square meters)
- Number of bedrooms
- Age of the building
- Location / neighborhood
- Distance to city center

> *"Same line-fitting idea — just admitting houses are complicated."*

In [4]:
# ============================================
# Demo: Simple vs Multiple Linear Regression
# ============================================

# Generate synthetic house price data
np.random.seed(42)
n_houses = 200

# Features
size = np.random.uniform(50, 200, n_houses)  # Square meters
bedrooms = np.random.randint(1, 6, n_houses)
age = np.random.uniform(0, 50, n_houses)  # Years
distance_to_center = np.random.uniform(0, 20, n_houses)  # km

# True relationship (in thousands of kr)
price = (500 +  # Base price
         30 * size +  # 30k per sqm
         100 * bedrooms +  # 100k per bedroom
         -5 * age +  # -5k per year of age
         -20 * distance_to_center +  # -20k per km from center
         np.random.normal(0, 200, n_houses))  # Noise

house_df = pd.DataFrame({
    'Size (sqm)': size,
    'Bedrooms': bedrooms,
    'Age (years)': age,
    'Distance to Center (km)': distance_to_center,
    'Price (1000 kr)': price
})

# Simple Linear Regression (only size)
X_simple = house_df[['Size (sqm)']]
y = house_df['Price (1000 kr)']

model_simple = LinearRegression()
model_simple.fit(X_simple, y)
y_pred_simple = model_simple.predict(X_simple)
r2_simple = r2_score(y, y_pred_simple)

# Multiple Linear Regression (all features)
X_multiple = house_df[['Size (sqm)', 'Bedrooms', 'Age (years)', 'Distance to Center (km)']]
model_multiple = LinearRegression()
model_multiple.fit(X_multiple, y)
y_pred_multiple = model_multiple.predict(X_multiple)
r2_multiple = r2_score(y, y_pred_multiple)

# Create comparison visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'<b>Simple LR: Size Only</b><br>R² = {r2_simple:.3f}',
        f'<b>Multiple LR: All Features</b><br>R² = {r2_multiple:.3f}'
    )
)

# Simple regression plot
fig.add_trace(
    go.Scatter(
        x=y, y=y_pred_simple,
        mode='markers',
        marker=dict(size=8, color='coral', opacity=0.6),
        name='Simple LR',
        hovertemplate='Actual: %{x:.0f}k<br>Predicted: %{y:.0f}k<extra></extra>'
    ),
    row=1, col=1
)

# Multiple regression plot
fig.add_trace(
    go.Scatter(
        x=y, y=y_pred_multiple,
        mode='markers',
        marker=dict(size=8, color='seagreen', opacity=0.6),
        name='Multiple LR',
        hovertemplate='Actual: %{x:.0f}k<br>Predicted: %{y:.0f}k<extra></extra>'
    ),
    row=1, col=2
)

# Add perfect prediction lines
line_range = [y.min(), y.max()]
for col in [1, 2]:
    fig.add_trace(
        go.Scatter(
            x=line_range, y=line_range,
            mode='lines',
            line=dict(color='black', dash='dash', width=2),
            name='Perfect Prediction',
            showlegend=(col == 1)
        ),
        row=1, col=col
    )

fig.update_xaxes(title_text='Actual Price (1000 kr)', row=1, col=1)
fig.update_xaxes(title_text='Actual Price (1000 kr)', row=1, col=2)
fig.update_yaxes(title_text='Predicted Price (1000 kr)', row=1, col=1)
fig.update_yaxes(title_text='Predicted Price (1000 kr)', row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>🏠 Predicted vs Actual House Prices</b><br><sup>Multiple features dramatically improve predictions!</sup>',
        font=dict(size=18)
    ),
    width=1000,
    height=500,
    showlegend=True
)

fig.show()

# Print coefficients
print("\n📊 Model Comparison:")
print("="*60)
print(f"\n🔹 Simple Linear Regression (Size only):")
print(f"   R² = {r2_simple:.4f}")
print(f"   Price = {model_simple.intercept_:.1f} + {model_simple.coef_[0]:.2f} × Size")

print(f"\n🔹 Multiple Linear Regression (All features):")
print(f"   R² = {r2_multiple:.4f}")
print(f"\n   Coefficients:")
for feature, coef in zip(X_multiple.columns, model_multiple.coef_):
    print(f"   • {feature}: {coef:.2f}")
print(f"   • Intercept: {model_multiple.intercept_:.2f}")

print(f"\n💡 Improvement: R² increased from {r2_simple:.3f} to {r2_multiple:.3f}")
print(f"   ({(r2_multiple - r2_simple) / r2_simple * 100:.1f}% improvement in explained variance!)")


📊 Model Comparison:

🔹 Simple Linear Regression (Size only):
   R² = 0.9569
   Price = 597.8 + 29.10 × Size

🔹 Multiple Linear Regression (All features):
   R² = 0.9770

   Coefficients:
   • Size (sqm): 29.87
   • Bedrooms: 95.58
   • Age (years): -3.17
   • Distance to Center (km): -24.23
   • Intercept: 521.84

💡 Improvement: R² increased from 0.957 to 0.977
   (2.1% improvement in explained variance!)


---

# 2. Multiple Regression Model

<a id='2-multiple-regression-model'></a>

## 📐 The Mathematical Model

### Model Statement

$$y = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_p x_p + \varepsilon$$

### Matrix Notation

$$\mathbf{y} = \mathbf{X}\boldsymbol{\theta} + \boldsymbol{\varepsilon}$$

Where:
- $\mathbf{y}$: $m \times 1$ vector of responses (what we're predicting)
- $\mathbf{X}$: $m \times (p + 1)$ design matrix (includes intercept column of 1s)
- $\boldsymbol{\theta}$: $(p + 1) \times 1$ vector of coefficients $[\theta_0, \theta_1, \ldots, \theta_p]^T$
- $\boldsymbol{\varepsilon}$: $m \times 1$ vector of errors

### Least Squares Objective

We minimize the **sum of squared residuals**:

$$\min_{\boldsymbol{\theta}} \|\mathbf{y} - \mathbf{X}\boldsymbol{\theta}\|^2 = \min_{\boldsymbol{\theta}} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

### Closed-Form Solution (Normal Equation)

$$\hat{\boldsymbol{\theta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

In [5]:
# ============================================
# Interactive 3D Visualization: Regression Plane
# ============================================

# Generate data for 2 predictors
np.random.seed(42)
n = 100

x1 = np.random.uniform(0, 10, n)
x2 = np.random.uniform(0, 10, n)

# True model: y = 2 + 3*x1 + 1.5*x2 + noise
y_3d = 2 + 3*x1 + 1.5*x2 + np.random.normal(0, 2, n)

# Fit the model
X_3d = np.column_stack([np.ones(n), x1, x2])
theta_hat = np.linalg.lstsq(X_3d, y_3d, rcond=None)[0]

# Create mesh for the plane
x1_mesh, x2_mesh = np.meshgrid(
    np.linspace(0, 10, 30),
    np.linspace(0, 10, 30)
)
y_mesh = theta_hat[0] + theta_hat[1]*x1_mesh + theta_hat[2]*x2_mesh

# Create the 3D plot
fig = go.Figure()

# Add the regression plane
fig.add_trace(go.Surface(
    x=x1_mesh, y=x2_mesh, z=y_mesh,
    colorscale='Blues',
    opacity=0.7,
    name='Regression Plane',
    showscale=False,
    hovertemplate='x₁: %{x:.1f}<br>x₂: %{y:.1f}<br>ŷ: %{z:.1f}<extra>Regression Plane</extra>'
))

# Add data points
fig.add_trace(go.Scatter3d(
    x=x1, y=x2, z=y_3d,
    mode='markers',
    marker=dict(size=5, color='red', opacity=0.8),
    name='Data Points',
    hovertemplate='x₁: %{x:.1f}<br>x₂: %{y:.1f}<br>y: %{z:.1f}<extra>Data Point</extra>'
))

# Add residual lines (for a subset of points)
y_pred_3d = theta_hat[0] + theta_hat[1]*x1 + theta_hat[2]*x2
for i in range(0, n, 10):  # Every 10th point
    fig.add_trace(go.Scatter3d(
        x=[x1[i], x1[i]],
        y=[x2[i], x2[i]],
        z=[y_3d[i], y_pred_3d[i]],
        mode='lines',
        line=dict(color='green', width=2),
        showlegend=(i == 0),
        name='Residuals'
    ))

fig.update_layout(
    title=dict(
        text='<b>📐 Multiple Linear Regression in 3D</b><br><sup>The regression "line" becomes a plane with 2 predictors</sup>',
        font=dict(size=18)
    ),
    scene=dict(
        xaxis_title='x₁',
        yaxis_title='x₂',
        zaxis_title='y',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, t=80, b=0)
)

fig.show()

print("\n📊 Fitted Model:")
print(f"   ŷ = {theta_hat[0]:.2f} + {theta_hat[1]:.2f}·x₁ + {theta_hat[2]:.2f}·x₂")
print(f"\n   True model: y = 2 + 3·x₁ + 1.5·x₂ + ε")
print(f"\n💡 The green lines show the residuals (prediction errors)")
print(f"   OLS minimizes the sum of squared lengths of these lines!")


📊 Fitted Model:
   ŷ = 1.82 + 2.93·x₁ + 1.64·x₂

   True model: y = 2 + 3·x₁ + 1.5·x₂ + ε

💡 The green lines show the residuals (prediction errors)
   OLS minimizes the sum of squared lengths of these lines!


## 🔑 Understanding "Holding Others Fixed"

### Partial Regression Coefficient

> **$\theta_j$ represents the change in $y$ for a one-unit increase in $x_j$, while all other predictors remain constant.**

This is the **key difference** from simple linear regression!

### Example: House Price Model

$$\text{Price} = \theta_0 + \theta_1 \cdot \text{Size} + \theta_2 \cdot \text{Age} + \theta_3 \cdot \text{Bedrooms} + \varepsilon$$

**Interpretation:**

| Coefficient | Interpretation |
|-------------|---------------|
| $\theta_1 = 3000$ kr/m² | Each additional sqm increases price by 3000 kr, *for houses of the same age and bedroom count* |
| $\theta_2 = -50000$ kr/year | Each additional year of age decreases price by 50,000 kr, *for houses of the same size and bedroom count* |
| $\theta_3 = 200000$ kr/bedroom | Each additional bedroom increases price by 200,000 kr, *for houses of the same size and age* |

In [6]:
# ============================================
# Demo: Effect of Adding a Confounder
# ============================================

# Scenario: Ice cream sales depend on temperature AND day type
np.random.seed(42)

# Generate data
n = 50

# Weekdays: lower temperature, lower sales
temp_weekday = np.random.normal(4, 1.5, n//2)
sales_weekday = 2 + 0.65 * temp_weekday + np.random.normal(0, 0.5, n//2)
is_weekend_weekday = np.zeros(n//2)

# Weekends: higher temperature, higher sales (plus weekend boost)
temp_weekend = np.random.normal(7, 1.5, n//2)
sales_weekend = 2 + 0.65 * temp_weekend + 2.0 + np.random.normal(0, 0.5, n//2)  # +2 weekend boost
is_weekend_weekend = np.ones(n//2)

# Combine data
temperature = np.concatenate([temp_weekday, temp_weekend])
sales = np.concatenate([sales_weekday, sales_weekend])
is_weekend = np.concatenate([is_weekend_weekday, is_weekend_weekend])
day_type = ['Weekend' if w else 'Weekday' for w in is_weekend]

confound_df = pd.DataFrame({
    'Temperature': temperature,
    'Sales': sales,
    'Is_Weekend': is_weekend,
    'Day_Type': day_type
})

# Model 1: Ignoring weekend (simple regression)
model1 = LinearRegression()
model1.fit(confound_df[['Temperature']], confound_df['Sales'])

# Model 2: Including weekend (multiple regression)
model2 = LinearRegression()
model2.fit(confound_df[['Temperature', 'Is_Weekend']], confound_df['Sales'])

# Create visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'<b>Model 1: Ignoring Day Type</b><br>Slope = {model1.coef_[0]:.2f}',
        f'<b>Model 2: Including Day Type</b><br>Temp Slope = {model2.coef_[0]:.2f}, Weekend Boost = {model2.coef_[1]:.2f}'
    )
)

# Model 1 (ignoring confounder)
fig.add_trace(
    go.Scatter(
        x=temperature, y=sales,
        mode='markers',
        marker=dict(size=10, color='gray', opacity=0.7),
        name='All Data',
        showlegend=True
    ),
    row=1, col=1
)

# Model 1 regression line
x_range = np.linspace(temperature.min(), temperature.max(), 100)
fig.add_trace(
    go.Scatter(
        x=x_range,
        y=model1.predict(x_range.reshape(-1, 1)),
        mode='lines',
        line=dict(color='red', width=3),
        name=f'Slope = {model1.coef_[0]:.2f} (WRONG!)'
    ),
    row=1, col=1
)

# Model 2 (including confounder)
colors = {'Weekday': 'royalblue', 'Weekend': 'orange'}

for day in ['Weekday', 'Weekend']:
    mask = confound_df['Day_Type'] == day
    fig.add_trace(
        go.Scatter(
            x=confound_df.loc[mask, 'Temperature'],
            y=confound_df.loc[mask, 'Sales'],
            mode='markers',
            marker=dict(size=10, color=colors[day], opacity=0.7,
                       symbol='circle' if day == 'Weekday' else 'square'),
            name=day
        ),
        row=1, col=2
    )

# Add separate regression lines for Model 2
for is_wknd, color, name in [(0, 'royalblue', 'Weekday'), (1, 'orange', 'Weekend')]:
    y_pred = model2.intercept_ + model2.coef_[0] * x_range + model2.coef_[1] * is_wknd
    fig.add_trace(
        go.Scatter(
            x=x_range, y=y_pred,
            mode='lines',
            line=dict(color=color, width=3),
            name=f'{name} fit'
        ),
        row=1, col=2
    )

fig.update_xaxes(title_text='Temperature (°C)', row=1, col=1)
fig.update_xaxes(title_text='Temperature (°C)', row=1, col=2)
fig.update_yaxes(title_text='Sales (units)', row=1, col=1)
fig.update_yaxes(title_text='Sales (units)', row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>🌡️ The Importance of Including Confounders</b><br><sup>Ignoring day type overestimates the temperature effect!</sup>',
        font=dict(size=18)
    ),
    width=1100,
    height=500,
    showlegend=True
)

fig.show()

print("\n🔍 Key Insight:")
print("="*60)
print(f"\nModel 1 (ignoring day type): Slope = {model1.coef_[0]:.3f}")
print(f"Model 2 (including day type): Slope = {model2.coef_[0]:.3f}")
print(f"\n⚠️  Model 1 overestimates the temperature effect by {((model1.coef_[0] - model2.coef_[0]) / model2.coef_[0] * 100):.0f}%!")
print(f"\n💡 Why? Weekends have BOTH higher temperatures AND more customers.")
print(f"   Model 1 wrongly attributes the weekend effect to temperature.")
print(f"\n   True temperature effect: {model2.coef_[0]:.2f} units/°C")
print(f"   Weekend boost: {model2.coef_[1]:.2f} units")


🔍 Key Insight:

Model 1 (ignoring day type): Slope = 1.024
Model 2 (including day type): Slope = 0.675

⚠️  Model 1 overestimates the temperature effect by 52%!

💡 Why? Weekends have BOTH higher temperatures AND more customers.
   Model 1 wrongly attributes the weekend effect to temperature.

   True temperature effect: 0.68 units/°C
   Weekend boost: 2.02 units


---

# 3. Interpretability & Correlated Predictors

<a id='3-interpretability--correlated-predictors'></a>

## 🎯 Best Case: Orthogonal Predictors

When predictors are **uncorrelated** (orthogonal):

- ✅ Clean credit assignment
- ✅ $\theta_j$ captures unique effect
- ✅ Stable coefficients
- ✅ Easy to interpret

**Mathematical property:** $\mathbf{X}^T\mathbf{X}$ is diagonal

Each predictor explains a **separate part** of $y$.

## ⚠️ Worst Case: Highly Correlated Predictors

When predictors are **correlated**:

- ❌ Shared variance ("Who gets credit?")
- ❌ Unstable coefficients
- ❌ Sign flips possible!
- ❌ Hard to interpret

**Extreme case:** If $x_2 = 2x_1$, there are infinitely many solutions!

### Multicollinearity

$\mathbf{X}^T\mathbf{X}$ nearly singular ⟹ $(\mathbf{X}^T\mathbf{X})^{-1}$ has large entries ⟹ **high coefficient variance**

In [7]:
# ============================================
# Interactive Demo: Effect of Correlation on Coefficient Stability
# ============================================

def generate_correlated_data(correlation, n=100, seed=42):
    """Generate two predictors with specified correlation."""
    np.random.seed(seed)
    
    # Generate correlated predictors
    mean = [0, 0]
    cov = [[1, correlation], [correlation, 1]]
    x1, x2 = np.random.multivariate_normal(mean, cov, n).T
    
    # True model: y = 2 + 3*x1 + 2*x2 + noise
    y = 2 + 3*x1 + 2*x2 + np.random.normal(0, 1, n)
    
    return x1, x2, y

def fit_and_get_coefficients(x1, x2, y, n_bootstrap=100):
    """Fit model and return coefficient estimates with bootstrap confidence."""
    X = np.column_stack([x1, x2])
    
    # Fit on full data
    model = LinearRegression()
    model.fit(X, y)
    
    # Bootstrap for uncertainty
    coefs_1, coefs_2 = [], []
    for _ in range(n_bootstrap):
        idx = np.random.choice(len(y), len(y), replace=True)
        model_boot = LinearRegression()
        model_boot.fit(X[idx], y[idx])
        coefs_1.append(model_boot.coef_[0])
        coefs_2.append(model_boot.coef_[1])
    
    return model.coef_, np.std(coefs_1), np.std(coefs_2), coefs_1, coefs_2

# Test different correlations
correlations = [0.0, 0.3, 0.6, 0.8, 0.95, 0.99]
results = []

for corr in correlations:
    x1, x2, y = generate_correlated_data(corr)
    coefs, std1, std2, boot_coefs1, boot_coefs2 = fit_and_get_coefficients(x1, x2, y)
    results.append({
        'correlation': corr,
        'theta1': coefs[0],
        'theta2': coefs[1],
        'std1': std1,
        'std2': std2,
        'boot_coefs1': boot_coefs1,
        'boot_coefs2': boot_coefs2
    })

# Create visualization
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'r = {r["correlation"]}' for r in results],
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

for idx, r in enumerate(results):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    # Plot bootstrap distributions
    fig.add_trace(
        go.Histogram(
            x=r['boot_coefs1'],
            name='θ₁',
            marker_color='royalblue',
            opacity=0.7,
            nbinsx=20,
            showlegend=(idx == 0)
        ),
        row=row, col=col
    )
    fig.add_trace(
        go.Histogram(
            x=r['boot_coefs2'],
            name='θ₂',
            marker_color='coral',
            opacity=0.7,
            nbinsx=20,
            showlegend=(idx == 0)
        ),
        row=row, col=col
    )
    
    # Add true value lines
    fig.add_vline(x=3, line_dash="dash", line_color="blue", row=row, col=col)
    fig.add_vline(x=2, line_dash="dash", line_color="red", row=row, col=col)

fig.update_layout(
    title=dict(
        text='<b>📊 How Correlation Affects Coefficient Stability</b><br><sup>Higher correlation → wider distributions → less reliable coefficients</sup>',
        font=dict(size=18)
    ),
    width=1100,
    height=700,
    barmode='overlay'
)

fig.show()

# Summary table
print("\n📊 Coefficient Stability Summary:")
print("="*70)
print(f"{'Correlation':>12} | {'θ₁ (true=3)':>14} | {'θ₂ (true=2)':>14} | {'Std(θ₁)':>10} | {'Std(θ₂)':>10}")
print("-"*70)
for r in results:
    print(f"{r['correlation']:>12.2f} | {r['theta1']:>14.3f} | {r['theta2']:>14.3f} | {r['std1']:>10.3f} | {r['std2']:>10.3f}")
print("\n⚠️  Notice how standard deviations EXPLODE as correlation approaches 1!")


📊 Coefficient Stability Summary:
 Correlation |    θ₁ (true=3) |    θ₂ (true=2) |    Std(θ₁) |    Std(θ₂)
----------------------------------------------------------------------
        0.00 |          3.191 |          1.828 |      0.100 |      0.122
        0.30 |          3.027 |          1.736 |      0.131 |      0.110
        0.60 |          3.086 |          1.701 |      0.158 |      0.137
        0.80 |          3.172 |          1.627 |      0.210 |      0.190
        0.95 |          3.448 |          1.359 |      0.400 |      0.381
        0.99 |          4.122 |          0.687 |      0.877 |      0.858

⚠️  Notice how standard deviations EXPLODE as correlation approaches 1!


In [8]:
# ============================================
# Visualization: Who Gets the Credit?
# ============================================

# When x1 ≈ x2, many different (θ1, θ2) combinations give similar predictions

# Create contour plot showing the constraint θ1 + θ2 = constant
theta1_range = np.linspace(-2, 12, 100)
theta2_range = np.linspace(-2, 12, 100)
T1, T2 = np.meshgrid(theta1_range, theta2_range)

# For highly correlated predictors, the loss function is elongated
# along θ1 + θ2 = constant
# Simulate a loss surface
loss = 0.5 * ((T1 - 5)**2 + (T2 - 5)**2) + 5 * ((T1 - 5) + (T2 - 5))**2 / 50

# The constraint line θ1 + θ2 = 10 (all equivalent solutions)
constraint_line = 10 - theta1_range

fig = go.Figure()

# Loss contours
fig.add_trace(go.Contour(
    x=theta1_range,
    y=theta2_range,
    z=loss,
    colorscale='Blues',
    contours=dict(start=0, end=50, size=5),
    showscale=False,
    name='Loss Contours',
    hovertemplate='θ₁: %{x:.1f}<br>θ₂: %{y:.1f}<br>Loss: %{z:.1f}<extra></extra>'
))

# Equivalent solutions line
fig.add_trace(go.Scatter(
    x=theta1_range,
    y=constraint_line,
    mode='lines',
    line=dict(color='red', width=3),
    name='θ₁ + θ₂ = 10 (equivalent solutions)'
))

# Mark specific solutions
solutions = [
    (10, 0, 'A: Only x₁ matters'),
    (0, 10, 'B: Only x₂ matters'),
    (5, 5, 'C: Both matter equally')
]

for t1, t2, label in solutions:
    fig.add_trace(go.Scatter(
        x=[t1], y=[t2],
        mode='markers+text',
        marker=dict(size=15, color='red', symbol='circle'),
        text=[label],
        textposition='top right',
        textfont=dict(size=12, color='red'),
        showlegend=False
    ))

fig.update_layout(
    title=dict(
        text='<b>🤔 Who Gets the Credit? The Ambiguity of Correlated Predictors</b><br><sup>When x₁ ≈ x₂, all points on the red line give similar predictions!</sup>',
        font=dict(size=18)
    ),
    xaxis_title='θ₁',
    yaxis_title='θ₂',
    width=800,
    height=700,
    xaxis=dict(range=[-2, 12]),
    yaxis=dict(range=[-2, 12], scaleanchor='x')
)

fig.show()

print("\n🔍 Key Insight: The 'Credit Assignment' Problem")
print("="*60)
print("\nWhen x₁ and x₂ are highly correlated:")
print("\n  Solution A: ŷ = 5 + 10·x₁ + 0·x₂  → 'Only x₁ matters'")
print("  Solution B: ŷ = 5 + 0·x₁ + 10·x₂  → 'Only x₂ matters'")
print("  Solution C: ŷ = 5 + 5·x₁ + 5·x₂   → 'Both matter equally'")
print("\n  All three give SIMILAR predictions!")
print("  But the INTERPRETATIONS are completely different!")
print("\n⚠️  This is why we can't trust individual coefficients")
print("    when predictors are highly correlated.")


🔍 Key Insight: The 'Credit Assignment' Problem

When x₁ and x₂ are highly correlated:

  Solution A: ŷ = 5 + 10·x₁ + 0·x₂  → 'Only x₁ matters'
  Solution B: ŷ = 5 + 0·x₁ + 10·x₂  → 'Only x₂ matters'
  Solution C: ŷ = 5 + 5·x₁ + 5·x₂   → 'Both matter equally'

  All three give SIMILAR predictions!
  But the INTERPRETATIONS are completely different!

⚠️  This is why we can't trust individual coefficients
    when predictors are highly correlated.


## 🎯 Prediction vs. Interpretation

### Key Distinction

**Prediction accuracy ≠ Interpretability**

| Aspect | Good for Prediction | Bad for Interpretation |
|--------|--------------------|-----------------------|
| Correlated predictors? | No problem! | Can't trust individual $\theta_j$ |
| Goal | Minimize $\sum(y_i - \hat{y}_i)^2$ | Understand "What is the effect of $x_j$?" |
| Multiple solutions | Many different $\boldsymbol{\theta}$ work | With correlation, question is ambiguous |

### Bottom Line

> *If your goal is just prediction, high correlation is not a problem.*  
> *If you want to understand effects, high correlation is very problematic!*

---

# 4. Categorical Variables in MLR

<a id='4-categorical-variables-in-mlr'></a>

## ⚠️ Why Categories Aren't Numeric

### The Problem

How do we include categorical variables like color, city, or day of week?

### ❌ Common Mistake

Assigning arbitrary numbers:
- Red = 1, Green = 2, Blue = 3

**This implies:**
- Blue is "3 times Red"
- Green is "halfway between" Red and Blue

**This is nonsense!**

### ✅ Correct Approach: Dummy Variables

Also called **indicator variables** or **one-hot encoding**:

| Color | IsRed | IsGreen |
|-------|-------|--------|
| Red   | 1     | 0      |
| Green | 0     | 1      |
| Blue  | 0     | 0      |

**Note:** We only need 2 dummies for 3 categories! (More on this next...)

In [9]:
# ============================================
# Demo: Dummy Variable Encoding
# ============================================

# Create sample data with day of week
np.random.seed(42)

days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
n_per_day = 20

# True effects (relative to Sunday as baseline)
day_effects = {
    'Monday': -100, 'Tuesday': -50, 'Wednesday': 0, 'Thursday': 50,
    'Friday': 150, 'Saturday': 300, 'Sunday': 0  # baseline
}

# Generate data
data = []
for day in days:
    base_sales = 1000  # Base Sunday sales
    sales = base_sales + day_effects[day] + np.random.normal(0, 50, n_per_day)
    for s in sales:
        data.append({'Day': day, 'Sales': s})

day_df = pd.DataFrame(data)

# Create dummy variables (drop Sunday as reference)
day_dummies = pd.get_dummies(day_df['Day'], prefix='Day', drop_first=False)
# Drop Sunday to use as reference
day_dummies = day_dummies.drop('Day_Sunday', axis=1)

# Fit model
X_days = day_dummies.values
y_days = day_df['Sales'].values

model_days = LinearRegression()
model_days.fit(X_days, y_days)

# Display encoding
print("📊 Dummy Variable Encoding (Reference: Sunday)")
print("="*80)
encoding_display = pd.DataFrame(index=days)
for col in day_dummies.columns:
    encoding_display[col.replace('Day_', '')] = [
        1 if day == col.replace('Day_', '') else 0 for day in days
    ]
print(encoding_display)

# Show coefficients
print("\n📈 Fitted Coefficients:")
print(f"  Intercept (Sunday baseline): {model_days.intercept_:.1f}")
for col, coef in zip(day_dummies.columns, model_days.coef_):
    day_name = col.replace('Day_', '')
    print(f"  {day_name}: {coef:+.1f} (Expected: {day_effects[day_name]:+.0f})")

📊 Dummy Variable Encoding (Reference: Sunday)
           Friday  Monday  Saturday  Thursday  Tuesday  Wednesday
Monday          0       1         0         0        0          0
Tuesday         0       0         0         0        1          0
Wednesday       0       0         0         0        0          1
Thursday        0       0         0         1        0          0
Friday          1       0         0         0        0          0
Saturday        0       0         1         0        0          0
Sunday          0       0         0         0        0          0

📈 Fitted Coefficients:
  Intercept (Sunday baseline): 994.3
  Friday: +154.5 (Expected: +150)
  Monday: -102.9 (Expected: -100)
  Saturday: +307.9 (Expected: +300)
  Thursday: +54.1 (Expected: +50)
  Tuesday: -57.6 (Expected: -50)
  Wednesday: +4.3 (Expected: +0)


In [10]:
# ============================================
# Visualization: Day-of-Week Effects
# ============================================

# Calculate predicted sales for each day
predicted_by_day = {'Sunday': model_days.intercept_}
for col, coef in zip(day_dummies.columns, model_days.coef_):
    day_name = col.replace('Day_', '')
    predicted_by_day[day_name] = model_days.intercept_ + coef

# Create visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Raw Data by Day', 'Coefficient Interpretation'),
    column_widths=[0.6, 0.4]
)

# Box plot of raw data
colors_days = px.colors.qualitative.Set2

for i, day in enumerate(days):
    day_data = day_df[day_df['Day'] == day]['Sales']
    fig.add_trace(
        go.Box(
            y=day_data,
            name=day[:3],
            marker_color=colors_days[i % len(colors_days)],
            boxpoints='outliers'
        ),
        row=1, col=1
    )

# Bar chart of coefficients
coef_df = pd.DataFrame({
    'Day': list(day_dummies.columns) + ['Day_Sunday'],
    'Coefficient': list(model_days.coef_) + [0]  # Sunday is reference (0)
})
coef_df['Day'] = coef_df['Day'].str.replace('Day_', '')

# Sort by day order
coef_df['sort_order'] = coef_df['Day'].map({day: i for i, day in enumerate(days)})
coef_df = coef_df.sort_values('sort_order')

fig.add_trace(
    go.Bar(
        x=coef_df['Day'].str[:3],
        y=coef_df['Coefficient'],
        marker_color=['coral' if c < 0 else 'seagreen' for c in coef_df['Coefficient']],
        text=[f"{c:+.0f}" for c in coef_df['Coefficient']],
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Day of Week', row=1, col=1)
fig.update_xaxes(title_text='Day of Week', row=1, col=2)
fig.update_yaxes(title_text='Sales (kr)', row=1, col=1)
fig.update_yaxes(title_text='Difference from Sunday (kr)', row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>📅 Interpreting Dummy Variable Coefficients</b><br><sup>Each coefficient shows the difference from the reference category (Sunday)</sup>',
        font=dict(size=18)
    ),
    width=1100,
    height=500,
    showlegend=False
)

# Add annotation
fig.add_annotation(
    x=0.95, y=0.95,
    xref='paper', yref='paper',
    text=f"Intercept (Sunday baseline):<br><b>{model_days.intercept_:.0f} kr</b>",
    showarrow=False,
    bgcolor='lightyellow',
    bordercolor='orange',
    borderwidth=2,
    font=dict(size=12)
)

fig.show()

print("\n💡 How to interpret:")
print(f"  • Expected Sunday sales: {model_days.intercept_:.0f} kr (intercept)")
print(f"  • Expected Saturday sales: {model_days.intercept_:.0f} + {model_days.coef_[5]:.0f} = {model_days.intercept_ + model_days.coef_[5]:.0f} kr")
print(f"  • Expected Monday sales: {model_days.intercept_:.0f} + ({model_days.coef_[1]:.0f}) = {model_days.intercept_ + model_days.coef_[1]:.0f} kr")


💡 How to interpret:
  • Expected Sunday sales: 994 kr (intercept)
  • Expected Saturday sales: 994 + 4 = 999 kr
  • Expected Monday sales: 994 + (-103) = 891 kr


## ⚠️ The Dummy Variable Trap

### Perfect Multicollinearity

If you include **all k dummies plus an intercept**, you create perfect multicollinearity!

**Why?** The sum of all dummies always equals 1:

$$\text{Mon} + \text{Tue} + \cdots + \text{Sun} = 1 \text{ (always)}$$

This is the **same as the intercept column** in $\mathbf{X}$!

### The Rule

> **k categories ⟹ k - 1 dummies**
>
> Always omit one category as the **reference**

$\mathbf{X}^T\mathbf{X}$ becomes singular ⟹ no unique solution!

In [ ]:
# ============================================
# Demo: The Dummy Variable Trap
# ============================================

# Create all dummies (including Sunday - THIS IS WRONG!)
all_dummies = pd.get_dummies(day_df['Day'], prefix='Day', drop_first=False)

print("❌ Including ALL dummies (Dummy Variable Trap):")
print("="*60)
print("\nSum of each row:")
print(all_dummies.sum(axis=1).head(10).values)  # All 1s!
print("\n⚠️ Every row sums to 1 - identical to the intercept column!")

# Show what happens
X_trap = sm.add_constant(all_dummies)
print(f"\n📊 Design matrix rank: {np.linalg.matrix_rank(X_trap.astype(float))}")
print(f"   Number of columns: {X_trap.shape[1]}")
print("\n💥 Rank < columns means X'X is SINGULAR (not invertible)!")

# Try to fit and show the warning
print("\n" + "="*60)
print("✅ Correct approach: Drop one dummy (use k-1 for k categories)")
print("="*60)

correct_dummies = pd.get_dummies(day_df['Day'], prefix='Day', drop_first=True)
X_correct = sm.add_constant(correct_dummies)
print(f"\n📊 Design matrix rank: {np.linalg.matrix_rank(X_correct.astype(float))}")
print(f"   Number of columns: {X_correct.shape[1]}")
print("\n✅ Rank = columns means X'X is invertible!")

❌ Including ALL dummies (Dummy Variable Trap):

Sum of each row:
[1 1 1 1 1 1 1 1 1 1]

⚠️ Every row sums to 1 - identical to the intercept column!


UFuncTypeError: Cannot cast ufunc 'svd' input from dtype('O') to dtype('float64') with casting rule 'same_kind'

In [ ]:
# ============================================
# Interactive Demo: Interactions with Categorical Variables
# ============================================

# Scenario: Study hours affect test scores differently for two groups
np.random.seed(42)
n_students = 100

# Generate data
study_hours = np.random.uniform(0, 6, n_students)
group = np.random.choice(['A', 'B'], n_students)
is_group_b = (group == 'B').astype(int)

# True model with interaction:
# Group A: score = 1 + 1*hours
# Group B: score = 2.5 + 2*hours  (different intercept AND slope!)
theta_0 = 1      # intercept for Group A
theta_1 = 1      # slope for Group A
theta_2 = 1.5    # additional intercept for Group B
theta_3 = 1      # additional slope for Group B

score = (theta_0 + theta_1 * study_hours + 
         theta_2 * is_group_b + 
         theta_3 * study_hours * is_group_b + 
         np.random.normal(0, 0.5, n_students))

interaction_df = pd.DataFrame({
    'Study_Hours': study_hours,
    'Group': group,
    'Is_Group_B': is_group_b,
    'Score': score
})

# Fit model with interaction
interaction_df['Interaction'] = interaction_df['Study_Hours'] * interaction_df['Is_Group_B']
X_interaction = interaction_df[['Study_Hours', 'Is_Group_B', 'Interaction']]
model_interaction = LinearRegression()
model_interaction.fit(X_interaction, score)

# Create visualization
fig = go.Figure()

colors_group = {'A': 'royalblue', 'B': 'coral'}
symbols_group = {'A': 'circle', 'B': 'triangle-up'}

for grp in ['A', 'B']:
    mask = interaction_df['Group'] == grp
    fig.add_trace(go.Scatter(
        x=interaction_df.loc[mask, 'Study_Hours'],
        y=interaction_df.loc[mask, 'Score'],
        mode='markers',
        marker=dict(size=10, color=colors_group[grp], symbol=symbols_group[grp], opacity=0.7),
        name=f'Group {grp}'
    ))

# Fitted lines
x_range = np.linspace(0, 6, 100)

# Group A: intercept + slope*x
y_A = model_interaction.intercept_ + model_interaction.coef_[0] * x_range
# Group B: (intercept + coef_B) + (slope + interaction)*x
y_B = (model_interaction.intercept_ + model_interaction.coef_[1] + 
       (model_interaction.coef_[0] + model_interaction.coef_[2]) * x_range)

fig.add_trace(go.Scatter(
    x=x_range, y=y_A,
    mode='lines',
    line=dict(color='royalblue', width=3),
    name=f'Group A: y = {model_interaction.intercept_:.2f} + {model_interaction.coef_[0]:.2f}x'
))

fig.add_trace(go.Scatter(
    x=x_range, y=y_B,
    mode='lines',
    line=dict(color='coral', width=3),
    name=f'Group B: y = {model_interaction.intercept_ + model_interaction.coef_[1]:.2f} + {model_interaction.coef_[0] + model_interaction.coef_[2]:.2f}x'
))

fig.update_layout(
    title=dict(
        text='<b>📚 Interaction: Different Slopes for Different Groups</b><br><sup>Studying helps Group B more than Group A (steeper slope)!</sup>',
        font=dict(size=18)
    ),
    xaxis_title='Study Hours',
    yaxis_title='Test Score',
    width=900,
    height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

print("\n📊 Model with Interaction:")
print("="*60)
print(f"\nFull model: Score = θ₀ + θ₁·Hours + θ₂·IsGroupB + θ₃·(Hours × IsGroupB)")
print(f"\n  θ₀ (intercept): {model_interaction.intercept_:.3f}")
print(f"  θ₁ (hours effect): {model_interaction.coef_[0]:.3f}")
print(f"  θ₂ (Group B intercept shift): {model_interaction.coef_[1]:.3f}")
print(f"  θ₃ (Group B slope shift): {model_interaction.coef_[2]:.3f}")
print(f"\n🔍 Interpretation:")
print(f"  • Group A baseline score: {model_interaction.intercept_:.2f}")
print(f"  • Group A: +{model_interaction.coef_[0]:.2f} score per study hour")
print(f"  • Group B baseline score: {model_interaction.intercept_ + model_interaction.coef_[1]:.2f}")
print(f"  • Group B: +{model_interaction.coef_[0] + model_interaction.coef_[2]:.2f} score per study hour")
print(f"\n💡 The interaction term captures that studying is MORE effective for Group B!")

---

# 5. Regularization: L2, L1, L0

<a id='5-regularization-l2-l1-l0'></a>

## 📉 The Problem: Overfitting

When we have many features, OLS can **overfit** the training data:

- Fits the noise, not just the signal
- Coefficients become very large and unstable
- Poor generalization to new data

## 💡 The Solution: Regularization

Add a **penalty** to the objective function:

$$\min_{\boldsymbol{\theta}} \underbrace{\|\mathbf{y} - \mathbf{X}\boldsymbol{\theta}\|^2}_{\text{Fit to data}} + \underbrace{\lambda \cdot \text{Penalty}(\boldsymbol{\theta})}_{\text{Complexity control}}$$

- $\lambda > 0$: Regularization strength (hyperparameter)
- Larger $\lambda$ ⟹ simpler model (smaller $|\theta_j|$)
- Smaller $\lambda$ ⟹ closer to OLS

In [ ]:
# ============================================
# Demo: Overfitting with Too Many Features
# ============================================

np.random.seed(42)

# Generate data: only 3 features are truly relevant out of 50
n_samples = 100
n_features = 50
n_relevant = 3

X_overfit = np.random.randn(n_samples, n_features)

# True coefficients: only first 3 are non-zero
true_coefs = np.zeros(n_features)
true_coefs[:n_relevant] = [3, -2, 1.5]

y_overfit = X_overfit @ true_coefs + np.random.randn(n_samples) * 2

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_overfit, y_overfit, test_size=0.3, random_state=42
)

# Fit OLS
ols_model = LinearRegression()
ols_model.fit(X_train, y_train)

# Predictions
y_train_pred = ols_model.predict(X_train)
y_test_pred = ols_model.predict(X_test)

# Calculate metrics
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Visualize coefficients
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('OLS Coefficients', 'True vs Estimated'),
    column_widths=[0.6, 0.4]
)

# Bar chart of coefficients
colors = ['seagreen' if i < n_relevant else 'coral' for i in range(n_features)]
fig.add_trace(
    go.Bar(
        x=list(range(n_features)),
        y=ols_model.coef_,
        marker_color=colors,
        name='OLS Coefficients'
    ),
    row=1, col=1
)

# Add true coefficients as markers
fig.add_trace(
    go.Scatter(
        x=list(range(n_features)),
        y=true_coefs,
        mode='markers',
        marker=dict(size=12, color='black', symbol='star'),
        name='True Coefficients'
    ),
    row=1, col=1
)

# Training vs Test MSE
fig.add_trace(
    go.Bar(
        x=['Training MSE', 'Test MSE'],
        y=[train_mse, test_mse],
        marker_color=['royalblue', 'coral'],
        text=[f'{train_mse:.2f}', f'{test_mse:.2f}'],
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='Feature Index', row=1, col=1)
fig.update_xaxes(title_text='', row=1, col=2)
fig.update_yaxes(title_text='Coefficient Value', row=1, col=1)
fig.update_yaxes(title_text='Mean Squared Error', row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>⚠️ Overfitting: OLS with Too Many Features</b><br><sup>Only 3 features are relevant (green), but OLS assigns non-zero weights to all!</sup>',
        font=dict(size=18)
    ),
    width=1100,
    height=500
)

fig.show()

print("\n📊 Overfitting Analysis:")
print("="*60)
print(f"\n🎯 True model: Only features 0, 1, 2 have non-zero coefficients")
print(f"   True values: [{true_coefs[0]:.1f}, {true_coefs[1]:.1f}, {true_coefs[2]:.1f}]")
print(f"   OLS estimates: [{ols_model.coef_[0]:.2f}, {ols_model.coef_[1]:.2f}, {ols_model.coef_[2]:.2f}]")
print(f"\n⚠️ Problem: OLS assigns non-zero weights to irrelevant features!")
print(f"   Sum of |coef| for irrelevant features: {np.sum(np.abs(ols_model.coef_[n_relevant:])):.2f}")
print(f"\n📉 Result: Training MSE = {train_mse:.2f}, Test MSE = {test_mse:.2f}")
print(f"   Test MSE is {test_mse/train_mse:.1f}x higher! (overfitting)")

## 📐 L2 Regularization (Ridge)

### Objective Function

$$\min_{\boldsymbol{\theta}} \|\mathbf{y} - \mathbf{X}\boldsymbol{\theta}\|^2 + \lambda\|\boldsymbol{\theta}\|_2^2 = \min_{\boldsymbol{\theta}} \sum_{i=1}^{m}(y_i - \hat{y}_i)^2 + \lambda\sum_{j=1}^{p}\theta_j^2$$

### Properties

- **Shrinks** all coefficients toward zero
- **Never** sets coefficients exactly to zero
- **Stabilizes** estimates when predictors are correlated
- **Closed-form solution:** $\boldsymbol{\theta}_{\text{ridge}} = (\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$

### Geometric Interpretation

The L2 penalty creates a **circular constraint** in coefficient space: $\theta_1^2 + \theta_2^2 \leq C$

In [ ]:
# ============================================
# Interactive Demo: Ridge Regression Geometry
# ============================================

# Create a visualization showing the L2 constraint geometry
theta1_range = np.linspace(-4, 6, 200)
theta2_range = np.linspace(-4, 6, 200)
T1, T2 = np.meshgrid(theta1_range, theta2_range)

# Loss function (elliptical contours centered at OLS solution)
ols_theta1, ols_theta2 = 3.5, 2.5  # Simulated OLS solution
loss = (T1 - ols_theta1)**2 + 0.5*(T2 - ols_theta2)**2 + 0.3*(T1 - ols_theta1)*(T2 - ols_theta2)

fig = go.Figure()

# Loss contours
fig.add_trace(go.Contour(
    x=theta1_range, y=theta2_range, z=loss,
    colorscale='Blues',
    contours=dict(start=0.5, end=20, size=2),
    showscale=False,
    name='OLS Loss',
    opacity=0.7
))

# L2 constraint circle
C = 5  # Constraint radius squared
theta = np.linspace(0, 2*np.pi, 100)
circle_x = np.sqrt(C) * np.cos(theta)
circle_y = np.sqrt(C) * np.sin(theta)

fig.add_trace(go.Scatter(
    x=circle_x, y=circle_y,
    mode='lines',
    line=dict(color='red', width=4),
    name='L2 Constraint (circle)',
    fill='toself',
    fillcolor='rgba(255, 0, 0, 0.1)'
))

# OLS solution
fig.add_trace(go.Scatter(
    x=[ols_theta1], y=[ols_theta2],
    mode='markers+text',
    marker=dict(size=15, color='blue', symbol='star'),
    text=['OLS'],
    textposition='top right',
    name='OLS Solution'
))

# Ridge solution (on the circle where contour is tangent)
# Approximate ridge solution
ridge_theta1 = 1.8
ridge_theta2 = 1.2

fig.add_trace(go.Scatter(
    x=[ridge_theta1], y=[ridge_theta2],
    mode='markers+text',
    marker=dict(size=15, color='red', symbol='circle'),
    text=['Ridge'],
    textposition='bottom left',
    name='Ridge Solution'
))

# Add arrow from OLS to Ridge
fig.add_annotation(
    x=ridge_theta1, y=ridge_theta2,
    ax=ols_theta1, ay=ols_theta2,
    xref='x', yref='y',
    axref='x', ayref='y',
    showarrow=True,
    arrowhead=3,
    arrowsize=1.5,
    arrowwidth=2,
    arrowcolor='gray'
)

fig.update_layout(
    title=dict(
        text='<b>⭕ Ridge Regression: Circular Constraint</b><br><sup>Ridge shrinks coefficients toward origin, but typically NOT to exactly zero</sup>',
        font=dict(size=18)
    ),
    xaxis_title='θ₁',
    yaxis_title='θ₂',
    width=800,
    height=700,
    xaxis=dict(range=[-4, 6], zeroline=True, zerolinewidth=2, zerolinecolor='gray'),
    yaxis=dict(range=[-4, 6], zeroline=True, zerolinewidth=2, zerolinecolor='gray', scaleanchor='x')
)

fig.show()

print("\n💡 Ridge Geometry:")
print("="*60)
print("• Blue contours: OLS loss function")
print("• Red circle: L2 constraint (θ₁² + θ₂² ≤ C)")
print("• OLS wants to be at the center of the contours")
print("• Ridge solution: where contour touches the circle")
print("• Note: Solution is typically NOT on an axis (no exact zeros)")

## 💎 L1 Regularization (Lasso)

### Objective Function

$$\min_{\boldsymbol{\theta}} \|\mathbf{y} - \mathbf{X}\boldsymbol{\theta}\|^2 + \lambda\|\boldsymbol{\theta}\|_1 = \min_{\boldsymbol{\theta}} \sum_{i=1}^{m}(y_i - \hat{y}_i)^2 + \lambda\sum_{j=1}^{p}|\theta_j|$$

### Properties

- **Shrinks** coefficients AND sets many **exactly to zero**
- Performs **automatic feature selection**
- Produces **sparse** models (few non-zero $\theta_j$)
- No closed-form solution (requires iterative optimization)

### Geometric Interpretation

The L1 penalty creates a **diamond constraint** in coefficient space: $|\theta_1| + |\theta_2| \leq C$

The **corners** of the diamond lie on the axes, making it likely for solutions to have some $\theta_j = 0$!

In [ ]:
# ============================================
# Interactive Demo: Lasso vs Ridge Geometry
# ============================================

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('L2 (Ridge): Circular Constraint', 'L1 (Lasso): Diamond Constraint'),
    horizontal_spacing=0.1
)

# Same loss contours for both
theta_range = np.linspace(-4, 6, 200)
T1, T2 = np.meshgrid(theta_range, theta_range)
ols_theta1, ols_theta2 = 3.5, 2.5
loss = (T1 - ols_theta1)**2 + 0.5*(T2 - ols_theta2)**2 + 0.3*(T1 - ols_theta1)*(T2 - ols_theta2)

# Add loss contours to both subplots
for col in [1, 2]:
    fig.add_trace(
        go.Contour(
            x=theta_range, y=theta_range, z=loss,
            colorscale='Blues',
            contours=dict(start=0.5, end=20, size=2),
            showscale=False,
            opacity=0.6
        ),
        row=1, col=col
    )

# L2 Circle
C = 5
theta = np.linspace(0, 2*np.pi, 100)
fig.add_trace(
    go.Scatter(
        x=np.sqrt(C) * np.cos(theta),
        y=np.sqrt(C) * np.sin(theta),
        mode='lines',
        line=dict(color='red', width=4),
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.15)',
        name='L2 Circle'
    ),
    row=1, col=1
)

# L1 Diamond
C_l1 = 2.5
diamond_x = [C_l1, 0, -C_l1, 0, C_l1]
diamond_y = [0, C_l1, 0, -C_l1, 0]
fig.add_trace(
    go.Scatter(
        x=diamond_x, y=diamond_y,
        mode='lines',
        line=dict(color='green', width=4),
        fill='toself',
        fillcolor='rgba(0, 255, 0, 0.15)',
        name='L1 Diamond'
    ),
    row=1, col=2
)

# OLS solution
for col in [1, 2]:
    fig.add_trace(
        go.Scatter(
            x=[ols_theta1], y=[ols_theta2],
            mode='markers',
            marker=dict(size=15, color='blue', symbol='star'),
            name='OLS',
            showlegend=(col == 1)
        ),
        row=1, col=col
    )

# Ridge solution (on circle)
fig.add_trace(
    go.Scatter(
        x=[1.8], y=[1.2],
        mode='markers+text',
        marker=dict(size=15, color='red', symbol='circle'),
        text=['Ridge'],
        textposition='bottom left',
        name='Ridge'
    ),
    row=1, col=1
)

# Lasso solution (on corner - one coefficient is zero!)
fig.add_trace(
    go.Scatter(
        x=[2.5], y=[0],
        mode='markers+text',
        marker=dict(size=15, color='green', symbol='diamond'),
        text=['Lasso (θ₂=0!)'],
        textposition='top right',
        name='Lasso'
    ),
    row=1, col=2
)

# Highlight the corners
fig.add_trace(
    go.Scatter(
        x=[C_l1, 0, -C_l1, 0],
        y=[0, C_l1, 0, -C_l1],
        mode='markers',
        marker=dict(size=12, color='green', symbol='square'),
        name='Corners (sparse solutions)'
    ),
    row=1, col=2
)

fig.update_xaxes(title_text='θ₁', row=1, col=1, range=[-4, 5], zeroline=True)
fig.update_xaxes(title_text='θ₁', row=1, col=2, range=[-4, 5], zeroline=True)
fig.update_yaxes(title_text='θ₂', row=1, col=1, range=[-4, 5], zeroline=True, scaleanchor='x')
fig.update_yaxes(title_text='θ₂', row=1, col=2, range=[-4, 5], zeroline=True, scaleanchor='x2')

fig.update_layout(
    title=dict(
        text='<b>💎 Ridge vs Lasso: Why Lasso Produces Zeros</b><br><sup>Lasso solution often lands at a corner (on an axis) → exact zeros!</sup>',
        font=dict(size=18)
    ),
    width=1100,
    height=600,
    showlegend=True
)

fig.show()

print("\n🔑 Key Difference:")
print("="*60)
print("\n• Ridge (L2): Circle has NO corners → solutions rarely at axes")
print("• Lasso (L1): Diamond HAS corners on axes → solutions often sparse!")
print("\n💡 This is why Lasso performs automatic feature selection!")

In [ ]:
# ============================================
# Compare Ridge vs Lasso on Real Data
# ============================================

# Use the overfitting data from before
lambdas = np.logspace(-3, 3, 50)

ridge_coefs = []
lasso_coefs = []

for lam in lambdas:
    # Ridge
    ridge = Ridge(alpha=lam)
    ridge.fit(X_train, y_train)
    ridge_coefs.append(ridge.coef_.copy())
    
    # Lasso
    lasso = Lasso(alpha=lam, max_iter=10000)
    lasso.fit(X_train, y_train)
    lasso_coefs.append(lasso.coef_.copy())

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

# Create regularization path plots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Ridge Regularization Path', 'Lasso Regularization Path'),
    horizontal_spacing=0.1
)

colors_path = px.colors.qualitative.Set1

# Plot first 10 coefficients for clarity
for i in range(10):
    color = colors_path[i % len(colors_path)]
    line_style = 'solid' if i < n_relevant else 'dash'
    name = f'θ{i} {"(relevant)" if i < n_relevant else "(noise)"}'
    
    # Ridge
    fig.add_trace(
        go.Scatter(
            x=lambdas, y=ridge_coefs[:, i],
            mode='lines',
            line=dict(color=color, width=2, dash=line_style),
            name=name,
            legendgroup=f'group{i}',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Lasso
    fig.add_trace(
        go.Scatter(
            x=lambdas, y=lasso_coefs[:, i],
            mode='lines',
            line=dict(color=color, width=2, dash=line_style),
            name=name,
            legendgroup=f'group{i}',
            showlegend=False
        ),
        row=1, col=2
    )

# Add zero line
for col in [1, 2]:
    fig.add_hline(y=0, line_dash='dot', line_color='gray', row=1, col=col)

fig.update_xaxes(type='log', title_text='λ (regularization strength)', row=1, col=1)
fig.update_xaxes(type='log', title_text='λ (regularization strength)', row=1, col=2)
fig.update_yaxes(title_text='Coefficient Value', row=1, col=1)
fig.update_yaxes(title_text='Coefficient Value', row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>📈 Regularization Paths: Ridge vs Lasso</b><br><sup>Solid lines = relevant features, Dashed = noise features</sup>',
        font=dict(size=18)
    ),
    width=1200,
    height=500,
    legend=dict(font=dict(size=10))
)

fig.show()

print("\n🔍 Key Observations:")
print("="*60)
print("\n📊 Ridge (L2):")
print("   • All coefficients shrink smoothly toward zero")
print("   • NO coefficients ever reach exactly zero")
print("   • All features remain in the model")
print("\n📊 Lasso (L1):")
print("   • Noise features (dashed) hit zero FIRST")
print("   • Relevant features (solid) persist longer")
print("   • Automatic feature selection!")

# Count zeros at a specific lambda
test_lambda_idx = 30
print(f"\n📌 At λ = {lambdas[test_lambda_idx]:.2f}:")
print(f"   Ridge zeros: {np.sum(np.abs(ridge_coefs[test_lambda_idx]) < 1e-6)}")
print(f"   Lasso zeros: {np.sum(np.abs(lasso_coefs[test_lambda_idx]) < 1e-6)}")

In [ ]:
# ============================================
# Interactive: Test MSE as a function of lambda
# ============================================

train_mse_ridge = []
test_mse_ridge = []
train_mse_lasso = []
test_mse_lasso = []

for lam in lambdas:
    # Ridge
    ridge = Ridge(alpha=lam)
    ridge.fit(X_train, y_train)
    train_mse_ridge.append(mean_squared_error(y_train, ridge.predict(X_train)))
    test_mse_ridge.append(mean_squared_error(y_test, ridge.predict(X_test)))
    
    # Lasso
    lasso = Lasso(alpha=lam, max_iter=10000)
    lasso.fit(X_train, y_train)
    train_mse_lasso.append(mean_squared_error(y_train, lasso.predict(X_train)))
    test_mse_lasso.append(mean_squared_error(y_test, lasso.predict(X_test)))

# Find optimal lambdas
best_ridge_idx = np.argmin(test_mse_ridge)
best_lasso_idx = np.argmin(test_mse_lasso)

fig = go.Figure()

# Ridge
fig.add_trace(go.Scatter(
    x=lambdas, y=train_mse_ridge,
    mode='lines',
    line=dict(color='royalblue', width=2, dash='dash'),
    name='Ridge Train MSE'
))
fig.add_trace(go.Scatter(
    x=lambdas, y=test_mse_ridge,
    mode='lines',
    line=dict(color='royalblue', width=3),
    name='Ridge Test MSE'
))

# Lasso
fig.add_trace(go.Scatter(
    x=lambdas, y=train_mse_lasso,
    mode='lines',
    line=dict(color='coral', width=2, dash='dash'),
    name='Lasso Train MSE'
))
fig.add_trace(go.Scatter(
    x=lambdas, y=test_mse_lasso,
    mode='lines',
    line=dict(color='coral', width=3),
    name='Lasso Test MSE'
))

# Mark optimal points
fig.add_trace(go.Scatter(
    x=[lambdas[best_ridge_idx]],
    y=[test_mse_ridge[best_ridge_idx]],
    mode='markers+text',
    marker=dict(size=15, color='royalblue', symbol='star'),
    text=[f'Best Ridge λ={lambdas[best_ridge_idx]:.2f}'],
    textposition='top right',
    name='Best Ridge'
))
fig.add_trace(go.Scatter(
    x=[lambdas[best_lasso_idx]],
    y=[test_mse_lasso[best_lasso_idx]],
    mode='markers+text',
    marker=dict(size=15, color='coral', symbol='star'),
    text=[f'Best Lasso λ={lambdas[best_lasso_idx]:.2f}'],
    textposition='top left',
    name='Best Lasso'
))

fig.update_layout(
    title=dict(
        text='<b>🎯 Finding the Optimal Regularization Strength</b><br><sup>The sweet spot balances underfitting (high λ) and overfitting (low λ)</sup>',
        font=dict(size=18)
    ),
    xaxis_title='λ (regularization strength)',
    yaxis_title='Mean Squared Error',
    xaxis_type='log',
    width=1000,
    height=600
)

fig.show()

print("\n📊 Optimal Regularization:")
print("="*60)
print(f"\n🔵 Ridge:")
print(f"   Best λ: {lambdas[best_ridge_idx]:.3f}")
print(f"   Test MSE: {test_mse_ridge[best_ridge_idx]:.3f}")
print(f"\n🔴 Lasso:")
print(f"   Best λ: {lambdas[best_lasso_idx]:.3f}")
print(f"   Test MSE: {test_mse_lasso[best_lasso_idx]:.3f}")
print(f"\n📌 OLS Test MSE (λ=0): {test_mse:.3f}")
print(f"\n💡 Regularization reduced test MSE by ~{(1 - min(test_mse_ridge[best_ridge_idx], test_mse_lasso[best_lasso_idx])/test_mse)*100:.0f}%!")

## 📊 Summary: L0, L1, L2 Comparison

| Property | L0 | L1 (Lasso) | L2 (Ridge) |
|----------|----|-----------|-----------|
| **Penalty** | $\|\boldsymbol{\theta}\|_0 = \sum I(\theta_j \neq 0)$ | $\|\boldsymbol{\theta}\|_1 = \sum|\theta_j|$ | $\|\boldsymbol{\theta}\|_2^2 = \sum\theta_j^2$ |
| **Sparsity** | Exact zeros | Many zeros | No exact zeros |
| **Solution** | Combinatorial (NP-hard) | Convex (efficient) | Closed-form |
| **Stability** | Unstable | Moderate | Very stable |
| **Interpretability** | Best | Good | Moderate |
| **Geometry** | Axes only | Diamond | Circle |

In [ ]:
# ============================================
# Visual: L0, L1, L2 Constraint Shapes
# ============================================

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('L0: Axes Only', 'L1: Diamond', 'L2: Circle')
)

t = np.linspace(-2, 2, 100)
theta_circ = np.linspace(0, 2*np.pi, 100)

# L0: Axes only
fig.add_trace(go.Scatter(x=[0, 0], y=[-2, 2], mode='lines', line=dict(color='blue', width=4), name='L0'), row=1, col=1)
fig.add_trace(go.Scatter(x=[-2, 2], y=[0, 0], mode='lines', line=dict(color='blue', width=4), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers', marker=dict(size=15, color='blue'), showlegend=False), row=1, col=1)

# L1: Diamond
C = 1.5
fig.add_trace(go.Scatter(
    x=[C, 0, -C, 0, C], y=[0, C, 0, -C, 0],
    mode='lines', line=dict(color='green', width=4),
    fill='toself', fillcolor='rgba(0,255,0,0.2)',
    name='L1'
), row=1, col=2)

# L2: Circle
fig.add_trace(go.Scatter(
    x=C*np.cos(theta_circ), y=C*np.sin(theta_circ),
    mode='lines', line=dict(color='orange', width=4),
    fill='toself', fillcolor='rgba(255,165,0,0.2)',
    name='L2'
), row=1, col=3)

for col in [1, 2, 3]:
    fig.update_xaxes(range=[-2, 2], zeroline=True, zerolinewidth=1, row=1, col=col)
    fig.update_yaxes(range=[-2, 2], zeroline=True, zerolinewidth=1, row=1, col=col, scaleanchor=f'x{col if col > 1 else ""}')

fig.update_layout(
    title=dict(
        text='<b>📐 Constraint Geometries for Different Penalties</b>',
        font=dict(size=18)
    ),
    width=1000,
    height=400,
    showlegend=False
)

fig.show()

---

# 6. Feature Scaling

<a id='6-feature-scaling'></a>

## 📏 Why Does Scaling Matter?

### Two Perspectives:

**1. Interpretation**
- Coefficient units depend on feature units
- Hard to compare magnitudes across features
- Example: 3000 kr/m² vs 50,000,000 kr/km² (same feature, different units!)

**2. Regularization**
- Penalties depend on $|\theta_j|$
- Unfair if features have different scales
- Features with large values → small coefficients → less penalty

### Bottom Line

> **Scaling is OPTIONAL for OLS, but ESSENTIAL for regularized regression!**

In [ ]:
# ============================================
# Demo: Impact of Scaling on Regularization
# ============================================

np.random.seed(42)
n = 200

# Feature 1: small scale [0, 1]
x1_unscaled = np.random.uniform(0, 1, n)
# Feature 2: large scale [0, 1000]
x2_unscaled = np.random.uniform(0, 1000, n)

# True model: both features are equally important
# y = 10*x1 + 0.01*x2 + noise  (coefficient magnitudes differ due to scale)
y_scale = 10*x1_unscaled + 0.01*x2_unscaled + np.random.normal(0, 0.5, n)

# Create DataFrames
df_unscaled = pd.DataFrame({'x1': x1_unscaled, 'x2': x2_unscaled})

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_unscaled)
df_scaled = pd.DataFrame(X_scaled, columns=['x1', 'x2'])

# Fit Lasso without scaling
lambdas_scale = np.logspace(-3, 1, 30)
coefs_unscaled = []
coefs_scaled = []

for lam in lambdas_scale:
    # Unscaled
    lasso_unscaled = Lasso(alpha=lam, max_iter=10000)
    lasso_unscaled.fit(df_unscaled, y_scale)
    coefs_unscaled.append(lasso_unscaled.coef_.copy())
    
    # Scaled
    lasso_scaled = Lasso(alpha=lam, max_iter=10000)
    lasso_scaled.fit(df_scaled, y_scale)
    coefs_scaled.append(lasso_scaled.coef_.copy())

coefs_unscaled = np.array(coefs_unscaled)
coefs_scaled = np.array(coefs_scaled)

# Create visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Lasso WITHOUT Scaling', 'Lasso WITH Scaling')
)

# Without scaling
fig.add_trace(go.Scatter(
    x=lambdas_scale, y=coefs_unscaled[:, 0],
    mode='lines', line=dict(color='royalblue', width=3),
    name='θ₁ (x1: [0,1])'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=lambdas_scale, y=coefs_unscaled[:, 1],
    mode='lines', line=dict(color='coral', width=3),
    name='θ₂ (x2: [0,1000])'
), row=1, col=1)

# With scaling
fig.add_trace(go.Scatter(
    x=lambdas_scale, y=coefs_scaled[:, 0],
    mode='lines', line=dict(color='royalblue', width=3),
    name='θ₁ (standardized)',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=lambdas_scale, y=coefs_scaled[:, 1],
    mode='lines', line=dict(color='coral', width=3),
    name='θ₂ (standardized)',
    showlegend=False
), row=1, col=2)

fig.add_hline(y=0, line_dash='dot', line_color='gray', row=1, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='gray', row=1, col=2)

fig.update_xaxes(type='log', title_text='λ', row=1, col=1)
fig.update_xaxes(type='log', title_text='λ', row=1, col=2)
fig.update_yaxes(title_text='Coefficient', row=1, col=1)
fig.update_yaxes(title_text='Coefficient', row=1, col=2)

fig.update_layout(
    title=dict(
        text='<b>⚖️ Why Scaling Matters for Regularization</b><br><sup>Both features are equally important, but without scaling, x2 gets "protected" from shrinkage!</sup>',
        font=dict(size=18)
    ),
    width=1100,
    height=500
)

fig.show()

print("\n🔍 Analysis:")
print("="*60)
print("\nTrue contribution to y:")
print(f"  x1: coefficient = 10, range [0,1] → max contribution = 10")
print(f"  x2: coefficient = 0.01, range [0,1000] → max contribution = 10")
print(f"  ⟹ Both features contribute EQUALLY!")
print("\n⚠️ Without scaling:")
print(f"  θ₁ appears ~{coefs_unscaled[0, 0] / coefs_unscaled[0, 1]:.0f}x larger than θ₂")
print(f"  Lasso penalizes θ₁ MORE → unfair!")
print(f"  x1 gets shrunk to zero first (WRONG!)")
print("\n✅ With scaling:")
print(f"  Both coefficients are comparable")
print(f"  Fair regularization!")

In [ ]:
# ============================================
# Visualization: Common Scaling Methods
# ============================================

np.random.seed(42)

# Create sample data with outliers
data_original = np.concatenate([np.random.normal(50, 10, 95), [100, 110, 120, 130, 140]])

# Apply scaling methods
# Z-score standardization
data_zscore = (data_original - np.mean(data_original)) / np.std(data_original)

# Min-Max normalization
data_minmax = (data_original - np.min(data_original)) / (np.max(data_original) - np.min(data_original))

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Original Data', 'Z-score Standardization', 'Min-Max Normalization')
)

fig.add_trace(go.Histogram(x=data_original, nbinsx=30, marker_color='royalblue', name='Original'), row=1, col=1)
fig.add_trace(go.Histogram(x=data_zscore, nbinsx=30, marker_color='seagreen', name='Z-score'), row=1, col=2)
fig.add_trace(go.Histogram(x=data_minmax, nbinsx=30, marker_color='coral', name='Min-Max'), row=1, col=3)

fig.update_xaxes(title_text='Value', row=1, col=1)
fig.update_xaxes(title_text='Value', row=1, col=2)
fig.update_xaxes(title_text='Value', row=1, col=3)

fig.update_layout(
    title=dict(
        text='<b>📊 Common Scaling Methods</b>',
        font=dict(size=18)
    ),
    width=1100,
    height=400,
    showlegend=False
)

fig.show()

print("\n📏 Scaling Methods Comparison:")
print("="*60)
print("\n1️⃣ Z-score Standardization: x' = (x - μ) / σ")
print(f"   Mean: {np.mean(data_zscore):.4f}, Std: {np.std(data_zscore):.4f}")
print(f"   Range: [{np.min(data_zscore):.2f}, {np.max(data_zscore):.2f}]")
print("   ✓ Mean = 0, Std = 1")
print("   ✓ Preserves outliers (good for detecting them)")
print("   ✓ Most common for regularization")

print("\n2️⃣ Min-Max Normalization: x' = (x - min) / (max - min)")
print(f"   Range: [{np.min(data_minmax):.4f}, {np.max(data_minmax):.4f}]")
print("   ✓ Range = [0, 1]")
print("   ⚠️ Sensitive to outliers")
print("   ✓ Good for bounded data (e.g., images)")

print("\n⚠️ Critical Rule:")
print("   ALWAYS fit scaler on TRAINING data only!")
print("   Then apply to test data using training statistics.")

---

# 7. Summary & Key Takeaways

<a id='7-summary--key-takeaways'></a>

## 📋 What We Learned

### 1. Partial Effects & "Holding Fixed"
- $\theta_j$ = effect of $x_j$ when **other predictors are constant**
- Only meaningful if you **can** hold others fixed
- Beware of confounders!

### 2. Correlation Hurts Interpretability
- Shared variance → unstable coefficients, sign flips
- **Prediction often OK**, but interpretation breaks down
- "Who gets credit?" becomes ambiguous

### 3. Categorical Variables
- Use **k - 1 dummy variables** for k categories
- One category is the **reference** (all dummies = 0)
- Coefficients = **difference from reference**

### 4. Scaling Matters for Regularization
- Coefficient units change with feature units
- **Penalties unfair** without scaling
- **Always scale before L1/L2!**

### 5. L2 = Stability, L1 = Sparsity, L0 = Best Subset
- **Ridge (L2):** shrinks all coefficients, very stable, no zeros
- **Lasso (L1):** sets many to zero, automatic feature selection
- **L0:** exact subset selection, but computationally hard

In [ ]:
# ============================================
# Final Interactive Summary
# ============================================

# Create a comprehensive comparison chart
methods = ['OLS', 'Ridge (L2)', 'Lasso (L1)']
properties = {
    'Handles correlated features': [1, 5, 3],  # 1=poor, 5=excellent
    'Feature selection': [1, 1, 5],
    'Interpretability': [4, 3, 4],
    'Computational speed': [5, 5, 4],
    'Stability': [2, 5, 3],
    'Requires scaling': [1, 5, 5],  # 1=no, 5=essential
}

fig = go.Figure()

colors_methods = ['#636EFA', '#EF553B', '#00CC96']

for i, method in enumerate(methods):
    values = [properties[prop][i] for prop in properties.keys()]
    values.append(values[0])  # Close the radar
    
    categories = list(properties.keys())
    categories.append(categories[0])
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name=method,
        line=dict(color=colors_methods[i]),
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 5])
    ),
    title=dict(
        text='<b>📊 Method Comparison Radar Chart</b><br><sup>Higher values = better/more important</sup>',
        font=dict(size=18)
    ),
    width=800,
    height=600,
    showlegend=True
)

fig.show()

print("\n" + "="*70)
print("                    🎓 KEY TAKEAWAYS")
print("="*70)
print("""
1. 📊 MULTIPLE REGRESSION extends simple regression to multiple predictors
   • Coefficients show partial effects ("holding others fixed")
   • Matrix notation: y = Xθ + ε

2. ⚠️  CORRELATION between predictors complicates interpretation
   • Predictions can still be good
   • But individual coefficients become unreliable

3. 🏷️  CATEGORICAL VARIABLES require dummy encoding
   • Use k-1 dummies for k categories (avoid the trap!)
   • Coefficients are differences from the reference

4. 📏 SCALING is essential for regularization
   • Without scaling, regularization is unfair to large-scale features
   • Always standardize before Ridge/Lasso

5. 🎯 REGULARIZATION controls model complexity
   • Ridge (L2): Shrinks all, never zeros, very stable
   • Lasso (L1): Produces zeros, automatic feature selection
   • Choose based on your goals (prediction vs interpretation)
""")
print("="*70)

---

## 🔮 What's Next?

After mastering Multiple Linear Regression, you'll be ready for:

1. **Statistical Inference for Regression**
   - Hypothesis testing for coefficients
   - Confidence intervals
   - Model diagnostics

2. **Model Selection & Validation**
   - Cross-validation
   - Information criteria (AIC, BIC)
   - Train/validation/test splits

3. **Beyond Linear Models**
   - Generalized Linear Models (GLMs)
   - Non-linear regression
   - Tree-based methods

---

## 📚 References

1. James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning*
2. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*
3. Course materials: TTK 4260 - Multivariate Data Analysis and ML, NTNU

---

**Happy Learning! 🚀**